# SOR

SPD methods solve f(x)=0.5*x.T@A@x-b.T@x. A small residual does not guarantee a small solution error. Stationary methods may diverge on an SPD matrix; inspect converged.

Attach only bcsstk17.mat using Add Input. MAT_PATH overrides discovery; no downloads are used. Sparse CSR CPU implementation; sequential sweeps or sparse reusable inner solves use NumPy/Numba/SciPy.


In [ ]:
import time
from pathlib import Path
import numpy as np
from scipy.io import loadmat
from scipy.sparse import csr_matrix, issparse

print("Sparse CPU implementation")


def data_path(name, path=None):
    if path is not None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    matches = sorted(Path("/kaggle/input").rglob(f"{name}.mat"))
    if not matches:
        p = Path("data") / f"{name}.mat"
        if p.is_file():
            return p
    if len(matches) != 1:
        raise FileNotFoundError(f"Attach {name}.mat or set MAT_PATH; matches: {matches}")
    return matches[0]



def relative_residual(A, x, b):
    return float(np.linalg.norm(A @ x - b) / max(np.linalg.norm(b), np.finfo(float).tiny))


def backward_substitution(U, b):
    x = np.zeros_like(b, dtype=np.float64)
    for i in range(len(b)-1, -1, -1):
        if U[i, i] == 0:
            raise ValueError("Zero pivot")
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def lu_solve(L, U, b, p=None):
    c = np.array(b if p is None else b[p], dtype=np.float64, copy=True)
    for i in range(len(c)):
        c[i] = (c[i] - L[i, :i] @ c[:i]) / L[i, i]
    return backward_substitution(U, c)


def iteration_info(A, x, b, k, tol):
    r = relative_residual(A, x, b)
    return {"iterations": k, "residual": r, "converged": bool(np.isfinite(r) and r <= tol)}


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from numba import njit


@njit
def _sweep(data, indices, indptr, d, b, x, omega):
    for i in range(len(b)):
        s = 0.0
        for j in range(indptr[i], indptr[i+1]):
            s += data[j] * x[indices[j]]
        x[i] += omega * (b[i] - s) / d[i]


def sor(A, b, omega=1.1, x0=None, tol=1e-8, maxiter=10000):
    omega = omega
    if not 0 < omega < 2:
        raise ValueError("Require 0 < omega < 2")
    C = csr_matrix(A, dtype=np.float64)
    C.sum_duplicates()
    d = C.diagonal()
    if np.any(d == 0):
        raise ValueError("Zero diagonal")
    x = np.zeros_like(b, dtype=np.float64) if x0 is None else np.array(x0, dtype=np.float64, copy=True)
    k = 0
    while k < maxiter:
        info = iteration_info(C, x, b, k, tol)
        if info["converged"] or not np.isfinite(info["residual"]):
            break
        _sweep(C.data, C.indices, C.indptr, d, b, x, omega)
        k += 1
    return x, iteration_info(C, x, b, k, tol)


## 3×3


In [ ]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = sor(A, b)
result, seconds = timed(sor, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
assert relative_residual(A, x, b) < 1e-8


## SuiteSparse: bcsstk17

Attach only bcsstk17.mat (10,974 × 10,974) using Add Input, or set MAT_PATH.

The matrix stays in sparse CSR storage. Inspect convergence status; the iteration limit may be reached.


In [ ]:
MATRIX = "bcsstk17"
MAT_PATH = None
A_raw = loadmat(data_path(MATRIX, MAT_PATH))["Problem"][0, 0]["A"].astype(np.float64)
A = csr_matrix(A_raw)
print("Matrix:", MATRIX, "shape:", A.shape)
x_true = np.ones(A.shape[0])
b = A @ x_true


In [ ]:
result, seconds = timed(sor, A, b)
print("Seconds:", seconds)
x, info = result
print(info)
print("Relative residual:", relative_residual(A, x, b))
print("Relative solution error:", float(np.linalg.norm(x-x_true) / np.linalg.norm(x_true)))
